In [1]:
# Setup: imports, device, seeds
import os
import math
import random
import json
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    import timm
except Exception as e:
    raise RuntimeError("Please install timm: pip install timm")

try:
    from sentence_transformers import SentenceTransformer
    SENTENCE_TFORMERS_AVAILABLE = True
except Exception:
    SENTENCE_TFORMERS_AVAILABLE = False

try:
    import faiss  # optional
    FAISS_AVAILABLE = True
except Exception:
    FAISS_AVAILABLE = False

try:
    from sklearn.metrics import f1_score
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


/home/saksham/anaconda3/envs/model_training/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
# Config
ROOT = "/home/saksham/coding/dl"
CSV_PATH = os.path.join(ROOT, "Dataset.csv")
IMAGES_ROOT = os.path.join(ROOT, "pooled_images")  # CSV paths like "images/..." are relative to pooled_images/

# Train on high-level categories (cookware, electronics, etc.)
# Model will also output fine-grained labels and attributes
CLASSIFICATION_MODE = 'coarse'  # Using coarse-grained training

# Training hyperparams (adjust as needed)
IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 0  # set >0 if numpy/torchvision available in worker processes
EPOCHS = 60  # for quick test; increase for better performance
BASE_LR = 1e-4
WEIGHT_DECAY = 5e-2
WARMUP_EPOCHS = 1
FREEZE_EPOCHS = 2  # freeze early for a few epochs

# Multitask loss weights
LAMBDA_ATTR = 0.5
LAMBDA_CONTR = 0.5
CONTR_TEMPERATURE = 0.07

# Retrieval embedding dimension
PROJ_DIM = 384

# Classification parameters - use ALL classes from dataset
USE_ALL_CLASSES = True  # Train on all classes in dataset
MIN_SAMPLES_PER_CLASS = 2  # Minimum samples per class for stratified split (lowered for inclusivity)

print({
    "csv": CSV_PATH,
    "use_all_classes": USE_ALL_CLASSES,
    "min_samples_per_class": MIN_SAMPLES_PER_CLASS,
    "epochs": EPOCHS,
    "bs": BATCH_SIZE,
    "img": IMG_SIZE,
})


{'csv': '/home/saksham/coding/dl/Dataset.csv', 'use_all_classes': True, 'min_samples_per_class': 2, 'epochs': 60, 'bs': 64, 'img': 224}


In [4]:
# DataFrame loading and vocab building

df = pd.read_csv(CSV_PATH)
if "Unnamed: 5" in df.columns:
    df = df.drop(columns=["Unnamed: 5"])  # empty column

print(f"📊 Loaded dataset: {len(df)} rows")

# Absolute image paths
def make_abs_path(rel_path: str) -> str:
    # Normalize Windows backslashes to forward slashes for cross-platform compatibility
    normalized_path = rel_path.replace("\\", "/")
    return os.path.join(IMAGES_ROOT, normalized_path)

df["abs_path"] = df["image_path"].apply(make_abs_path)

# Extract coarse (high-level) categories for training
def extract_coarse_class(label: str) -> str:
    """Extract high-level category (cookware, electronics, etc.)."""
    if not isinstance(label, str):
        return "unknown"
    label_clean = label.strip().lower().replace("-", "_")
    parts = [p for p in label_clean.split("_") if p]
    if not parts:
        return label_clean or "unknown"
    # handle patterns like "sports item" encoded as sports_item
    if len(parts) >= 2 and parts[0] in {"sports", "home", "daily"} and parts[1] in {"item", "items", "needs"}:
        return f"{parts[0]} {parts[1]}".strip()
    return parts[0]

# Apply coarse extraction for training
print("📦 Extracting COARSE categories from class labels...")
df["target_class"] = df["class_label"].apply(extract_coarse_class)

# Keep original fine-grained label for prediction/display
df["fine_label"] = df["class_label"].str.strip().str.lower().str.replace("-", "_")

# Filter classes based on minimum sample requirement
class_counts = df["target_class"].value_counts()
rare_classes = class_counts[class_counts < MIN_SAMPLES_PER_CLASS]
if not rare_classes.empty:
    print(f"\n⚠️  Classes with fewer than {MIN_SAMPLES_PER_CLASS} samples (will be excluded):")
    for cls_name, count in rare_classes.items():
        print(f"  {cls_name}: {count}")

eligible_classes = class_counts[class_counts >= MIN_SAMPLES_PER_CLASS].index.tolist()

if USE_ALL_CLASSES:
    # Use all eligible classes
    selected_classes = eligible_classes
    print(f"\n✅ Using ALL {len(selected_classes)} eligible classes (>= {MIN_SAMPLES_PER_CLASS} samples each)")
else:
    # Legacy mode: top-K classes (kept for backward compatibility)
    NUM_CLASSES_TARGET = 10
    selected_classes = eligible_classes[:NUM_CLASSES_TARGET]
    print(f"\n✅ Using top {len(selected_classes)} classes by frequency")

if not selected_classes:
    raise ValueError(f"No classes meet the minimum sample requirement ({MIN_SAMPLES_PER_CLASS}); please lower MIN_SAMPLES_PER_CLASS or inspect the dataset.")

df = df[df["target_class"].isin(selected_classes)].reset_index(drop=True)
print(f"\n📋 Selected classes ({len(selected_classes)}):", selected_classes[:10], "..." if len(selected_classes) > 10 else "")
print(f"📈 Filtered dataset size: {len(df)} rows")
print(f"\nPer-class distribution (top 20):")
print(df["target_class"].value_counts().head(20))

# ========================================
# DYNAMIC ATTRIBUTE PARSING
# Discover all unique attribute keys from the data
# ========================================
print("\n" + "="*60)
print("🔍 DYNAMIC ATTRIBUTE DISCOVERY")
print("="*60)

def parse_attributes_dynamic(attr_str: str) -> Dict[str, str]:
    """Dynamically parse all attributes from string without hardcoding keys."""
    values = {}
    if isinstance(attr_str, str) and attr_str.strip():
        parts = [p.strip() for p in attr_str.split(";") if p.strip()]
        for p in parts:
            if ":" in p:
                k, v = p.split(":", 1)
                k = k.strip().lower()
                v = v.strip().lower()
                values[k] = v if v else "unknown"
    return values

# Alias for backward compatibility
parse_attributes = parse_attributes_dynamic

# Parse all attributes
attr_rows = df["attributes"].apply(parse_attributes_dynamic)

# Discover all unique attribute keys
all_facet_keys = set()
for row in attr_rows:
    all_facet_keys.update(row.keys())

FACETS = sorted(all_facet_keys)
print(f"\n📌 Discovered {len(FACETS)} unique attribute facets:")
print(f"   {FACETS}")

# Normalize: ensure all rows have all facets (fill with 'unknown')
def normalize_attributes(attr_dict: Dict[str, str]) -> Dict[str, str]:
    return {facet: attr_dict.get(facet, "unknown") for facet in FACETS}

attr_rows = attr_rows.apply(normalize_attributes)

# Build facet vocabularies (include 'unknown')
facet_to_values: Dict[str, List[str]] = {}
for facet in FACETS:
    vals = sorted(set([row[facet] for row in attr_rows]))
    if "unknown" not in vals:
        vals = ["unknown"] + vals
    facet_to_values[facet] = vals

# Class label maps
# Coarse classes (for stratified split and primary classification)
coarse_classes = sorted(set(df["target_class"]))
coarse_to_id = {c: i for i, c in enumerate(coarse_classes)}
id_to_coarse = {i: c for c, i in coarse_to_id.items()}

# Fine-grained classes (all unique fine labels)
fine_classes = sorted(set(df["fine_label"]))
fine_to_id = {c: i for i, c in enumerate(fine_classes)}
id_to_fine = {i: c for c, i in fine_to_id.items()}

print(f"Coarse classes: {len(coarse_classes)}")
print(f"Fine-grained classes: {len(fine_classes)}")

from pathlib import Path
DATASET_TAG = f"{Path(CSV_PATH).stem}_{len(df)}_{len(coarse_classes)}c_{len(fine_classes)}f"
print("Checkpoint dataset tag:", DATASET_TAG)
# Keep backward compatibility
classes = coarse_classes
cls_to_id = coarse_to_id
id_to_cls = id_to_coarse

# Facet value maps
facet_to_id: Dict[str, Dict[str, int]] = {
    facet: {v: i for i, v in enumerate(values)} for facet, values in facet_to_values.items()
}
id_to_facet_value: Dict[str, Dict[int, str]] = {
    facet: {i: v for v, i in mapping.items()} for facet, mapping in facet_to_id.items()
}

print("Facet vocab sizes:")
for facet in FACETS:
    print(f"  {facet}: {len(facet_to_values[facet])} values -> {facet_to_values[facet]}")

# ========================================
# UNIFIED ATTRIBUTE APPROACH
# Treat class labels as additional attributes
# ========================================
print("\n" + "="*60)
print("🔄 UNIFIED ATTRIBUTE APPROACH")
print("   Treating class labels as attributes")
print("="*60)

# Extended facets including class labels
FACETS_UNIFIED = ["coarse_class", "fine_class", "color", "material", "condition", "size"]

# Build unified facet vocabularies
facet_to_values_unified: Dict[str, List[str]] = {
    "coarse_class": coarse_classes,
    "fine_class": fine_classes,
    "color": facet_to_values["color"],
    "material": facet_to_values["material"],
    "condition": facet_to_values["condition"],
    "size": facet_to_values["size"]
}

facet_to_id_unified: Dict[str, Dict[str, int]] = {
    facet: {v: i for i, v in enumerate(values)} 
    for facet, values in facet_to_values_unified.items()
}

id_to_facet_value_unified: Dict[str, Dict[int, str]] = {
    facet: {i: v for v, i in mapping.items()} 
    for facet, mapping in facet_to_id_unified.items()
}

print("\nUnified facet vocab sizes:")
for facet in FACETS_UNIFIED:
    print(f"  {facet}: {len(facet_to_values_unified[facet])} values")
print("="*60)

# ========================================
# HIERARCHICAL MAPPING: Coarse -> Fine Classes
# ========================================
print("\n🔗 Building hierarchical class mapping...")

# Build mapping from coarse class to valid fine classes
coarse_to_fine_mapping: Dict[str, List[str]] = {}
for coarse_class in coarse_classes:
    # Find all fine classes that belong to this coarse class
    fine_in_coarse = set()
    for fine_class in fine_classes:
        # Extract coarse part from fine class (e.g., "travel_bag" -> "travel")
        fine_coarse = extract_coarse_class(fine_class)
        if fine_coarse == coarse_class:
            fine_in_coarse.add(fine_class)
    coarse_to_fine_mapping[coarse_class] = sorted(fine_in_coarse)

print("\nHierarchical class structure:")
for coarse, fines in coarse_to_fine_mapping.items():
    print(f"  {coarse}: {len(fines)} fine classes")
    if len(fines) <= 5:
        print(f"    → {fines}")
    else:
        print(f"    → {fines[:3]} ... (+{len(fines)-3} more)")

# Create mapping from coarse_id to valid fine_ids
coarse_id_to_fine_ids: Dict[int, List[int]] = {}
for coarse_class, coarse_id in coarse_to_id.items():
    fine_classes_list = coarse_to_fine_mapping[coarse_class]
    fine_ids_list = [fine_to_id[fc] for fc in fine_classes_list]
    coarse_id_to_fine_ids[coarse_id] = fine_ids_list

print("\n✅ Hierarchical mapping created!")
print("="*60)

# ========================================
# VALIDATE IMAGE PATHS
# ========================================
validate_image_paths(df, images_root=IMAGES_ROOT, sample_show=5, sample_check=100)


📊 Loaded dataset: 11880 rows
📦 Extracting COARSE categories from class labels...

✅ Using ALL 64 eligible classes (>= 2 samples each)

📋 Selected classes (64): ['clothing', 'personal', 'pen', 'tableware', 'electronics', 'stationery', 'footwear', 'travel', 'backpack', 'mouse'] ...
📈 Filtered dataset size: 11880 rows

Per-class distribution (top 20):
target_class
clothing       706
personal       672
pen            536
tableware      450
electronics    401
stationery     364
footwear       360
travel         360
backpack       360
mouse          358
stationary     355
sports         325
notebook       320
earphones      303
keyboard       300
sneakers       300
food           285
medical        281
bathroom       248
toothpaste     246
Name: count, dtype: int64

🔍 DYNAMIC ATTRIBUTE DISCOVERY

📌 Discovered 11 unique attribute facets:
   ['"color', 'color', 'colour', 'condition', 'conidition', 'features', 'material', 'pattern', 'shape', 'size', 'style']
Coarse classes: 64
Fine-grained clas

(11880, 0)

In [5]:
# Dataset and transforms
from PIL import Image
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class MultiTaskDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, facet_to_id: Dict[str, Dict[str, int]], cls_to_id: Dict[str, int], fine_to_id: Dict[str, int], transform=None):
        self.frame = frame.reset_index(drop=True)
        self.facet_to_id = facet_to_id
        self.cls_to_id = cls_to_id
        self.fine_to_id = fine_to_id
        self.transform = transform

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int):
        row = self.frame.iloc[idx]
        image = Image.open(row["abs_path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        coarse_id = self.cls_to_id[row["target_class"]]
        fine_id = self.fine_to_id[row["fine_label"]]
        attrs = parse_attributes(row["attributes"])
        facet_ids = {facet: self.facet_to_id[facet][attrs.get(facet, "unknown")] for facet in FACETS}
        caption = row.get("caption", "")
        return image, coarse_id, fine_id, facet_ids, caption, row.get("instance_id", "")

# Custom collate function to properly handle dictionary of facet_ids
def collate_multitask(batch):
    images = torch.stack([item[0] for item in batch])
    coarse_ids = [item[1] for item in batch]
    fine_ids = [item[2] for item in batch]
    facet_ids = [item[3] for item in batch]  # list of dicts
    captions = [item[4] for item in batch]
    inst_ids = [item[5] for item in batch]
    return images, coarse_ids, fine_ids, facet_ids, captions, inst_ids

# Train/Val/Test split (stratified by coarse class)
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=SEED, stratify=df["target_class"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, stratify=temp_df["target_class"])

train_set = MultiTaskDataset(train_df, facet_to_id, cls_to_id, fine_to_id, transform=train_transforms)
val_set = MultiTaskDataset(val_df, facet_to_id, cls_to_id, fine_to_id, transform=val_transforms)
test_set = MultiTaskDataset(test_df, facet_to_id, cls_to_id, fine_to_id, transform=val_transforms)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_multitask)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_multitask)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_multitask)

print(f"DataLoaders created with num_workers={NUM_WORKERS}")
print(f"Dataset sizes: train={len(train_set)}, val={len(val_set)}, test={len(test_set)}")
len(train_set), len(val_set), len(test_set)


DataLoaders created with num_workers=0
Dataset sizes: train=8316, val=1782, test=1782


(8316, 1782, 1782)

In [6]:
# ========================================
# UNIFIED ATTRIBUTE MODEL - Classes as Attributes
# ========================================

# Dataset for unified approach
class UnifiedAttributeDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, facet_to_id_unified: Dict[str, Dict[str, int]], transform=None):
        self.frame = frame.reset_index(drop=True)
        self.facet_to_id = facet_to_id_unified
        self.transform = transform

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int):
        row = self.frame.iloc[idx]
        image = Image.open(row["abs_path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        
        # Parse all attributes including class labels
        attrs = parse_attributes(row["attributes"])
        facet_ids = {
            "coarse_class": self.facet_to_id["coarse_class"][row["target_class"]],
            "fine_class": self.facet_to_id["fine_class"][row["fine_label"]],
            "color": self.facet_to_id["color"][attrs.get("color", "unknown")],
            "material": self.facet_to_id["material"][attrs.get("material", "unknown")],
            "condition": self.facet_to_id["condition"][attrs.get("condition", "unknown")],
            "size": self.facet_to_id["size"][attrs.get("size", "unknown")]
        }
        
        caption = row.get("caption", "")
        return image, facet_ids, caption, row.get("instance_id", "")

# Collate function for unified approach
def collate_unified(batch):
    images = torch.stack([item[0] for item in batch])
    facet_ids = [item[1] for item in batch]  # list of dicts
    captions = [item[2] for item in batch]
    inst_ids = [item[3] for item in batch]
    return images, facet_ids, captions, inst_ids

# Create unified datasets
train_set_unified = UnifiedAttributeDataset(train_df, facet_to_id_unified, transform=train_transforms)
val_set_unified = UnifiedAttributeDataset(val_df, facet_to_id_unified, transform=val_transforms)
test_set_unified = UnifiedAttributeDataset(test_df, facet_to_id_unified, transform=val_transforms)

train_loader_unified = DataLoader(train_set_unified, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_unified)
val_loader_unified = DataLoader(val_set_unified, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_unified)
test_loader_unified = DataLoader(test_set_unified, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_unified)

print(f"✅ Unified DataLoaders created")
print(f"   Dataset sizes: train={len(train_set_unified)}, val={len(val_set_unified)}, test={len(test_set_unified)}")


✅ Unified DataLoaders created
   Dataset sizes: train=8316, val=1782, test=1782


In [7]:
# Unified Attribute Model - Only attribute heads, no separate classification
class UnifiedAttributeModel(nn.Module):
    def __init__(self, backbone_name: str, facet_to_id: Dict[str, Dict[str, int]], proj_dim: int = 384):
        super().__init__()
        self.backbone_name = backbone_name
        
        # Create backbone
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0, global_pool='avg')
        embed_dim = getattr(self.backbone, 'num_features', 192)
        self.embed_dim = embed_dim
        self.proj_dim = proj_dim

        # Only attribute heads - treating classes as attributes
        self.attr_heads = nn.ModuleDict({})
        for facet, mapping in facet_to_id.items():
            self.attr_heads[facet] = nn.Linear(embed_dim, len(mapping))

        # Image projection for retrieval
        self.img_proj = nn.Sequential(
            nn.Linear(embed_dim, proj_dim),
            nn.ReLU(inplace=True),
            nn.Linear(proj_dim, proj_dim)
        )

    def forward(self, x: torch.Tensor):
        feats = self.backbone(x)
        logits_attrs = {facet: head(feats) for facet, head in self.attr_heads.items()}
        img_emb = self.img_proj(feats)
        img_emb = F.normalize(img_emb, dim=-1)
        return feats, logits_attrs, img_emb

# Create unified model
model_unified = UnifiedAttributeModel(
    backbone_name='deit_tiny_patch16_224',
    facet_to_id=facet_to_id_unified,
    proj_dim=PROJ_DIM
).to(device)

print(f"✅ Unified model created")
print(f"   Backbone: DeiT-Tiny")
print(f"   Embed dim: {model_unified.embed_dim}")
print(f"   Attribute heads: {len(model_unified.attr_heads)}")
print(f"   Head sizes: ", {k: v.out_features for k, v in model_unified.attr_heads.items()})


Unexpected keys (norm.bias, norm.weight) found while loading pretrained weights. This may be expected if model is being adapted.


✅ Unified model created
   Backbone: DeiT-Tiny
   Embed dim: 192
   Attribute heads: 6
   Head sizes:  {'coarse_class': 64, 'fine_class': 186, 'color': 100, 'material': 35, 'condition': 15, 'size': 5}


In [8]:
# Losses, optimizer, and utilities

def cosine_warmup_lr(epoch, base_lr, warmup_epochs, total_epochs):
    if epoch < warmup_epochs:
        return base_lr * float(epoch + 1) / float(warmup_epochs)
    progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


def compute_losses(logits_coarse, coarse_targets, logits_fine, fine_targets, logits_attrs: Dict[str, torch.Tensor], facet_targets: Dict[str, torch.Tensor]):
    loss_coarse = F.cross_entropy(logits_coarse, coarse_targets)
    loss_fine = F.cross_entropy(logits_fine, fine_targets)
    loss_attr = 0.0
    for facet, logits in logits_attrs.items():
        loss_attr = loss_attr + F.cross_entropy(logits, facet_targets[facet])
    loss_attr = loss_attr / max(1, len(logits_attrs))
    return loss_coarse, loss_fine, loss_attr


def contrastive_loss(img_emb: torch.Tensor, txt_emb: torch.Tensor, temperature: float = 0.07):
    img_emb = F.normalize(img_emb, dim=-1)
    txt_emb = F.normalize(txt_emb, dim=-1)
    logits = img_emb @ txt_emb.t() / temperature  # [B, B]
    targets = torch.arange(img_emb.size(0), device=img_emb.device)
    loss_i2t = F.cross_entropy(logits, targets)
    loss_t2i = F.cross_entropy(logits.t(), targets)
    return (loss_i2t + loss_t2i) * 0.5


# Optimizer
# Optimizer (legacy multi-head; only if `model` exists)
if 'model' in globals():
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
    frozen = False
else:
    # Unified models use optimizer_unified / optimizer_unified_tinyvit elsewhere
    optimizer = None
    scaler = None
    frozen = False


def set_backbone_trainable(module: nn.Module, trainable: bool):
    for p in module.parameters():
        p.requires_grad = trainable

# Metrics
@torch.no_grad()
def accuracy(pred: torch.Tensor, target: torch.Tensor) -> float:
    pred_labels = pred.argmax(dim=1)
    return (pred_labels == target).float().mean().item()

@torch.no_grad()
def macro_f1(pred: torch.Tensor, target: torch.Tensor) -> float:
    if not SKLEARN_AVAILABLE:
        return accuracy(pred, target)
    y_true = target.cpu().numpy()
    y_pred = pred.argmax(dim=1).cpu().numpy()
    return float(f1_score(y_true, y_pred, average='macro'))


In [10]:
# ✅ Dataset-scoped checkpoint helpers
from pathlib import Path

if 'CHECKPOINT_DIR' not in globals():
    CHECKPOINT_DIR = os.path.join(ROOT, 'checkpoints')

def _ckpt_subdir(dataset_tag: str) -> str:
    sub = os.path.join(CHECKPOINT_DIR, dataset_tag)
    Path(sub).mkdir(parents=True, exist_ok=True)
    return sub

def save_checkpoint(model, optimizer, epoch, metrics, model_name="model", best=False, dataset_tag: str = None):
    dataset_tag = dataset_tag or globals().get('DATASET_TAG', 'default')
    subdir = _ckpt_subdir(dataset_tag)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict() if optimizer is not None else None,
        'metrics': metrics,
        'model_config': {
            'backbone_name': getattr(model, 'backbone_name', None),
            'embed_dim': getattr(model, 'embed_dim', None),
            'proj_dim': getattr(model, 'proj_dim', None),
            'dataset_tag': dataset_tag,
            'model_name': model_name,
        }
    }
    path = os.path.join(subdir, f"{model_name}_best.pt" if best else f"{model_name}_epoch_{epoch}.pt")
    torch.save(checkpoint, path)
    if best:
        print(f"💾 Saved best to {path}")
    return path

def load_checkpoint(model, optimizer=None, model_name="model", dataset_tag: str = None):
    dataset_tag = dataset_tag or globals().get('DATASET_TAG', 'default')
    subdir = _ckpt_subdir(dataset_tag)
    best_path = os.path.join(subdir, f"{model_name}_best.pt")
    if os.path.exists(best_path):
        print(f"📂 Loading checkpoint from {best_path}")
        checkpoint = torch.load(best_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        if optimizer is not None and checkpoint.get('optimizer_state_dict') is not None:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        print(f"✅ Loaded epoch {checkpoint.get('epoch')} | metrics: {checkpoint.get('metrics')}")
        return True, checkpoint.get('epoch', 0), checkpoint.get('metrics', {})
    print(f"❌ No checkpoint at {best_path}")
    return False, 0, {}

def checkpoint_exists(model_name="model", dataset_tag: str = None):
    dataset_tag = dataset_tag or globals().get('DATASET_TAG', 'default')
    subdir = _ckpt_subdir(dataset_tag)
    return os.path.exists(os.path.join(subdir, f"{model_name}_best.pt"))

print(f"Checkpoint root: {CHECKPOINT_DIR} | dataset tag: {globals().get('DATASET_TAG','default')}")

Checkpoint root: /home/saksham/coding/dl/checkpoints | dataset tag: Dataset_11880_64c_186f


In [11]:
# Training and evaluation for unified approach

# Optimizer for unified model
optimizer_unified = torch.optim.AdamW(model_unified.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
scaler_unified = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

@torch.no_grad()
def evaluate_unified(model: nn.Module, loader: DataLoader, facet_list: List[str], use_hierarchy: bool = True) -> Dict[str, float]:
    """Evaluate unified model treating classes as attributes."""
    model.eval()
    facet_accs = {facet: [] for facet in facet_list}
    facet_f1s = {facet: [] for facet in facet_list}
    fine_hier_accs = []  # Fine class accuracy with hierarchical constraint
    
    for images, facet_ids, captions, _ in loader:
        images = images.to(device, non_blocking=True)
        facet_batch = {f: torch.as_tensor([d[f] for d in facet_ids], device=device) for f in facet_list}
        
        with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
            _, logits_attrs, _ = model(images)
        
        # Evaluate all facets normally
        for f in facet_list:
            facet_accs[f].append(accuracy(logits_attrs[f], facet_batch[f]))
            facet_f1s[f].append(macro_f1(logits_attrs[f], facet_batch[f]))
        
        # Evaluate fine class with hierarchical constraint
        if use_hierarchy:
            coarse_preds = logits_attrs["coarse_class"].argmax(dim=1)
            fine_probs = torch.softmax(logits_attrs["fine_class"], dim=1)
            fine_targets = facet_batch["fine_class"]
            
            correct = 0
            for i in range(len(coarse_preds)):
                coarse_id = coarse_preds[i].item()
                
                if coarse_id in coarse_id_to_fine_ids:
                    valid_fine_ids = coarse_id_to_fine_ids[coarse_id]
                    
                    # Mask and renormalize
                    masked_probs = torch.zeros_like(fine_probs[i])
                    masked_probs[valid_fine_ids] = fine_probs[i][valid_fine_ids]
                    
                    if masked_probs.sum() > 0:
                        fine_pred = masked_probs.argmax().item()
                    else:
                        fine_pred = fine_probs[i].argmax().item()
                else:
                    fine_pred = fine_probs[i].argmax().item()
                
                if fine_pred == fine_targets[i].item():
                    correct += 1
            
            fine_hier_accs.append(correct / len(coarse_preds))
    
    out = {}
    for f in facet_list:
        out[f'{f}_acc'] = float(np.mean(facet_accs[f])) if facet_accs[f] else 0.0
        out[f'{f}_f1'] = float(np.mean(facet_f1s[f])) if facet_f1s[f] else 0.0
    
    # Add hierarchical fine class accuracy
    if use_hierarchy and fine_hier_accs:
        out['fine_class_acc_hier'] = float(np.mean(fine_hier_accs))
    
    # Overall average
    all_accs = [out[f'{f}_acc'] for f in facet_list]
    out['avg_acc'] = float(np.mean(all_accs))
    
    return out

def hierarchical_consistency_loss(coarse_logits: torch.Tensor, fine_logits: torch.Tensor, 
                                   coarse_targets: torch.Tensor, fine_targets: torch.Tensor) -> torch.Tensor:
    """
    Compute hierarchical consistency loss.
    Encourages the model to only predict fine classes that are valid for the coarse class.
    """
    batch_size = coarse_logits.size(0)
    fine_probs = torch.softmax(fine_logits, dim=1)
    
    # Create a mask for valid fine classes given coarse class
    valid_mask = torch.zeros_like(fine_probs)
    
    for i in range(batch_size):
        coarse_id = coarse_targets[i].item()
        if coarse_id in coarse_id_to_fine_ids:
            valid_fine_ids = coarse_id_to_fine_ids[coarse_id]
            valid_mask[i, valid_fine_ids] = 1.0
    
    # Penalize probability mass on invalid fine classes
    invalid_prob_mass = (fine_probs * (1 - valid_mask)).sum(dim=1).mean()
    
    return invalid_prob_mass


def train_unified(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, 
                  optimizer, scaler, epochs: int = EPOCHS, model_name: str = "unified_model",
                  use_contrastive: bool = True, use_hierarchical_loss: bool = True):
    """Train unified model."""
    frozen = False
    best_val_avg_acc = 0.0
    
    for epoch in range(epochs):
        model.train()
        
        # Freeze/unfreeze strategy
        if (epoch < FREEZE_EPOCHS) and not frozen:
            set_backbone_trainable(model.backbone, False)
            for n, m in model.backbone.named_modules():
                if 'blocks' in n or 'fc' in n or 'head' in n:
                    set_backbone_trainable(m, True)
            for p in model.attr_heads.parameters(): p.requires_grad = True
            for p in model.img_proj.parameters(): p.requires_grad = True
        elif (epoch == FREEZE_EPOCHS) and not frozen:
            set_backbone_trainable(model.backbone, True)
            frozen = True

        # Adjust LR
        lr = cosine_warmup_lr(epoch, BASE_LR, WARMUP_EPOCHS, epochs)
        for g in optimizer.param_groups:
            g['lr'] = lr

        running = {"loss": 0.0, "loss_attr": 0.0, "loss_ctr": 0.0, "loss_hier": 0.0}
        prog_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
        
        for images, facet_ids, captions, _ in prog_bar:
            images = images.to(device, non_blocking=True)
            facet_batch = {f: torch.as_tensor([d[f] for d in facet_ids], device=device) for f in FACETS_UNIFIED}

            with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
                _, logits_attrs, img_emb = model(images)
                
                # All attributes have equal weight
                loss_attr = 0.0
                for facet, logits in logits_attrs.items():
                    loss_attr = loss_attr + F.cross_entropy(logits, facet_batch[facet])
                loss_attr = loss_attr / len(logits_attrs)
                
                # Hierarchical consistency loss
                if use_hierarchical_loss:
                    loss_hier = hierarchical_consistency_loss(
                        logits_attrs["coarse_class"],
                        logits_attrs["fine_class"],
                        facet_batch["coarse_class"],
                        facet_batch["fine_class"]
                    )
                else:
                    loss_hier = torch.tensor(0.0, device=device)
                
                # Contrastive loss
                if use_contrastive and text_encoder is not None:
                    txt_emb = text_encoder.encode(list(captions))
                    loss_ctr = contrastive_loss(img_emb, txt_emb, temperature=CONTR_TEMPERATURE)
                else:
                    loss_ctr = torch.tensor(0.0, device=device)
                
                # Total loss
                loss = loss_attr + 0.5 * loss_hier + LAMBDA_CONTR * loss_ctr

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            running["loss"] += loss.item()
            running["loss_attr"] += loss_attr.item()
            running["loss_ctr"] += float(loss_ctr.item())
            running["loss_hier"] += float(loss_hier.item())

            prog_bar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "attr": f"{loss_attr.item():.4f}",
                "hier": f"{float(loss_hier.item()):.4f}",
                "ctr": f"{float(loss_ctr.item()):.4f}",
            })

        n_batches = max(1, len(train_loader))
        log = {k: v / n_batches for k, v in running.items()}
        val_metrics = evaluate_unified(model, val_loader, FACETS_UNIFIED, use_hierarchy=use_hierarchical_loss)
        
        print(f"Epoch {epoch+1:03d}/{epochs} LR {lr:.2e} | ", end="")
        print("train:", {k: round(v, 4) for k, v in log.items()})
        
        fine_acc_str = f"fine={val_metrics['fine_class_acc']:.4f}"
        if 'fine_class_acc_hier' in val_metrics:
            fine_acc_str += f"(hier={val_metrics['fine_class_acc_hier']:.4f})"
        
        print(f"  val: coarse={val_metrics['coarse_class_acc']:.4f} {fine_acc_str} " +
              f"color={val_metrics['color_acc']:.4f} material={val_metrics['material_acc']:.4f} " +
              f"avg={val_metrics['avg_acc']:.4f}")

        # Save if best
        if val_metrics['avg_acc'] > best_val_avg_acc:
            best_val_avg_acc = val_metrics['avg_acc']
            save_checkpoint(model, optimizer, epoch+1, val_metrics, model_name=model_name, best=True,dataset_tag=DATASET_TAG)
    
    print(f"\n✅ Training complete! Best avg val acc: {best_val_avg_acc:.4f}")
    return model

print("✅ Training functions ready for unified model")


✅ Training functions ready for unified model


/tmp/ipykernel_25863/2698944262.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_unified = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))


In [12]:
# 🏋️ Load Unified DeiT Model (Classes as Attributes)

# Hardcoded checkpoint path
CHECKPOINT_PATH_DEIT = "/home/saksham/coding/dl/checkpoints/Dataset_11880_64c_186f/deit_unified_attr_best.pt"

print(f"📝 Loading DeiT model from: {CHECKPOINT_PATH_DEIT}")

# Load checkpoint
checkpoint = torch.load(CHECKPOINT_PATH_DEIT, map_location=device)
model_unified.load_state_dict(checkpoint['model_state_dict'])
if checkpoint.get('optimizer_state_dict') is not None:
    optimizer_unified.load_state_dict(checkpoint['optimizer_state_dict'])

print(f"✅ Loaded checkpoint from epoch {checkpoint.get('epoch')}")
print(f"   Metrics: {checkpoint.get('metrics')}")


📝 Loading DeiT model from: /home/saksham/coding/dl/checkpoints/Dataset_11880_64c_186f/deit_unified_attr_best.pt
✅ Loaded checkpoint from epoch 51
   Metrics: {'coarse_class_acc': 0.8419518853936877, 'coarse_class_f1': 0.7670491546170873, 'fine_class_acc': 0.8368262244122369, 'fine_class_f1': 0.7128291096581079, 'color_acc': 0.7778191140719822, 'color_f1': 0.5585681114765652, 'material_acc': 0.8150628315550941, 'material_f1': 0.6514040783382636, 'condition_acc': 0.8109499003205981, 'condition_f1': 0.6581430434917211, 'size_acc': 0.893353174839701, 'size_f1': 0.7746880173403097, 'fine_class_acc_hier': 0.8255621693121693, 'avg_acc': 0.8293271884322168}


In [13]:
# Helpers required by evaluate_unified
try:
    from sklearn.metrics import f1_score
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

@torch.no_grad()
def accuracy(pred: torch.Tensor, target: torch.Tensor) -> float:
    return (pred.argmax(dim=1) == target).float().mean().item()

@torch.no_grad()
def macro_f1(pred: torch.Tensor, target: torch.Tensor) -> float:
    if not SKLEARN_AVAILABLE:
        return accuracy(pred, target)
    y_true = target.detach().cpu().numpy()
    y_pred = pred.argmax(dim=1).detach().cpu().numpy()
    return float(f1_score(y_true, y_pred, average='macro'))

# Evaluate unified model on test set
test_metrics_unified = evaluate_unified(model_unified, test_loader_unified, FACETS_UNIFIED, use_hierarchy=True)

print("\n" + "="*70)
print("📊 UNIFIED MODEL TEST RESULTS (Classes as Attributes)")
print("="*70)
print(f"\n🎯 Classification Performance:")
print(f"   Coarse Class Accuracy: {test_metrics_unified['coarse_class_acc']:.4f}")
print(f"   Coarse Class F1:       {test_metrics_unified['coarse_class_f1']:.4f}")
print(f"   Fine Class Accuracy:   {test_metrics_unified['fine_class_acc']:.4f}")
print(f"   Fine Class F1:         {test_metrics_unified['fine_class_f1']:.4f}")
if 'fine_class_acc_hier' in test_metrics_unified:
    print(f"   Fine Class (Hierarchical): {test_metrics_unified['fine_class_acc_hier']:.4f} ⭐")

print(f"\n🏷️  Attribute Performance:")
print(f"   Color Accuracy:        {test_metrics_unified['color_acc']:.4f}  (F1: {test_metrics_unified['color_f1']:.4f})")
print(f"   Material Accuracy:     {test_metrics_unified['material_acc']:.4f}  (F1: {test_metrics_unified['material_f1']:.4f})")
print(f"   Condition Accuracy:    {test_metrics_unified['condition_acc']:.4f}  (F1: {test_metrics_unified['condition_f1']:.4f})")
print(f"   Size Accuracy:         {test_metrics_unified['size_acc']:.4f}  (F1: {test_metrics_unified['size_f1']:.4f})")

print(f"\n⭐ Overall Average Accuracy: {test_metrics_unified['avg_acc']:.4f}")
print("="*70)

/tmp/ipykernel_25863/2698944262.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
/home/saksham/anaconda3/envs/model_training/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/home/saksham/anaconda3/envs/model_training/lib/python3.11/site-packages/sklearn/metrics/_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")
/home/saksham/anaconda3/envs/model_training/lib/python3.11/site-packages/sklearn/utils/multiclass.py:79: UserWarnin


📊 UNIFIED MODEL TEST RESULTS (Classes as Attributes)

🎯 Classification Performance:
   Coarse Class Accuracy: 0.8580
   Coarse Class F1:       0.7951
   Fine Class Accuracy:   0.8452
   Fine Class F1:         0.7229
   Fine Class (Hierarchical): 0.8385 ⭐

🏷️  Attribute Performance:
   Color Accuracy:        0.7721  (F1: 0.5361)
   Material Accuracy:     0.8203  (F1: 0.6637)
   Condition Accuracy:    0.8123  (F1: 0.6730)
   Size Accuracy:         0.8781  (F1: 0.7283)

⭐ Overall Average Accuracy: 0.8310


/tmp/ipykernel_25863/2698944262.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
/home/saksham/anaconda3/envs/model_training/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/home/saksham/anaconda3/envs/model_training/lib/python3.11/site-packages/sklearn/metrics/_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")
/home/saksham/anaconda3/envs/model_training/lib/python3.11/site-packages/sklearn/utils/multiclass.py:79: UserWarnin

In [14]:
# 🔄 CREATE UNIFIED TINYVIT MODEL

# Create unified TinyVIT model
model_unified_tinyvit = UnifiedAttributeModel(
    backbone_name='tiny_vit_5m_224.dist_in22k_ft_in1k',
    facet_to_id=facet_to_id_unified,
    proj_dim=PROJ_DIM
).to(device)

print(f"✅ Unified TinyVIT model created")
print(f"   Backbone: TinyVIT-5M")
print(f"   Embed dim: {model_unified_tinyvit.embed_dim}")
print(f"   Attribute heads: {len(model_unified_tinyvit.attr_heads)}")
print(f"   Head sizes: ", {k: v.out_features for k, v in model_unified_tinyvit.attr_heads.items()})

# Optimizer for unified TinyVIT
optimizer_unified_tinyvit = torch.optim.AdamW(model_unified_tinyvit.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
scaler_unified_tinyvit = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

print("✅ Optimizer and scaler ready for unified TinyVIT")


✅ Unified TinyVIT model created
   Backbone: TinyVIT-5M
   Embed dim: 320
   Attribute heads: 6
   Head sizes:  {'coarse_class': 64, 'fine_class': 186, 'color': 100, 'material': 35, 'condition': 15, 'size': 5}
✅ Optimizer and scaler ready for unified TinyVIT


/tmp/ipykernel_25863/3508383550.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_unified_tinyvit = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))


In [15]:
# 🏋️ Load Unified TinyVIT Model

# Hardcoded checkpoint path
CHECKPOINT_PATH_TINYVIT = "/home/saksham/coding/dl/checkpoints/Dataset_11880_64c_186f/tinyvit_unified_attr_best.pt"

print(f"📝 Loading TinyVIT model from: {CHECKPOINT_PATH_TINYVIT}")

# Load checkpoint
checkpoint = torch.load(CHECKPOINT_PATH_TINYVIT, map_location=device)
model_unified_tinyvit.load_state_dict(checkpoint['model_state_dict'])
if checkpoint.get('optimizer_state_dict') is not None:
    optimizer_unified_tinyvit.load_state_dict(checkpoint['optimizer_state_dict'])

print(f"✅ Loaded checkpoint from epoch {checkpoint.get('epoch')}")
print(f"   Metrics: {checkpoint.get('metrics')}")


📝 Loading TinyVIT model from: /home/saksham/coding/dl/checkpoints/Dataset_11880_64c_186f/tinyvit_unified_attr_best.pt
✅ Loaded checkpoint from epoch 58
   Metrics: {'coarse_class_acc': 0.8762607468026025, 'coarse_class_f1': 0.8134854677550607, 'fine_class_acc': 0.8733672286782946, 'fine_class_f1': 0.7762526551845135, 'color_acc': 0.7512400788920266, 'color_f1': 0.46956155964935126, 'material_acc': 0.8078703710011074, 'material_f1': 0.6105075335383365, 'condition_acc': 0.802682706287929, 'condition_f1': 0.6455767768784139, 'size_acc': 0.8654513891254153, 'size_f1': 0.6949657851789005, 'fine_class_acc_hier': 0.855510085978836, 'avg_acc': 0.8294787534645627}


In [16]:
# Evaluate unified TinyVIT on test set
test_metrics_tinyvit = evaluate_unified(model_unified_tinyvit, test_loader_unified, FACETS_UNIFIED, use_hierarchy=True)

print("\n" + "="*70)
print("📊 UNIFIED TINYVIT TEST RESULTS")
print("="*70)
print(f"\n🎯 Classification Performance:")
print(f"   Coarse Class Accuracy: {test_metrics_tinyvit['coarse_class_acc']:.4f}")
print(f"   Coarse Class F1:       {test_metrics_tinyvit['coarse_class_f1']:.4f}")
print(f"   Fine Class Accuracy:   {test_metrics_tinyvit['fine_class_acc']:.4f}")
print(f"   Fine Class F1:         {test_metrics_tinyvit['fine_class_f1']:.4f}")
if 'fine_class_acc_hier' in test_metrics_tinyvit:
    print(f"   Fine Class (Hierarchical): {test_metrics_tinyvit['fine_class_acc_hier']:.4f} ⭐")

print(f"\n🏷️  Attribute Performance:")
print(f"   Color Accuracy:        {test_metrics_tinyvit['color_acc']:.4f}  (F1: {test_metrics_tinyvit['color_f1']:.4f})")
print(f"   Material Accuracy:     {test_metrics_tinyvit['material_acc']:.4f}  (F1: {test_metrics_tinyvit['material_f1']:.4f})")
print(f"   Condition Accuracy:    {test_metrics_tinyvit['condition_acc']:.4f}  (F1: {test_metrics_tinyvit['condition_f1']:.4f})")
print(f"   Size Accuracy:         {test_metrics_tinyvit['size_acc']:.4f}  (F1: {test_metrics_tinyvit['size_f1']:.4f})")

print(f"\n⭐ Overall Average Accuracy: {test_metrics_tinyvit['avg_acc']:.4f}")
print("="*70)


/tmp/ipykernel_25863/2698944262.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
/home/saksham/anaconda3/envs/model_training/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/home/saksham/anaconda3/envs/model_training/lib/python3.11/site-packages/sklearn/metrics/_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")
/home/saksham/anaconda3/envs/model_training/lib/python3.11/site-packages/sklearn/utils/multiclass.py:79: UserWarnin


📊 UNIFIED TINYVIT TEST RESULTS

🎯 Classification Performance:
   Coarse Class Accuracy: 0.8836
   Coarse Class F1:       0.8323
   Fine Class Accuracy:   0.8832
   Fine Class F1:         0.7912
   Fine Class (Hierarchical): 0.8618 ⭐

🏷️  Attribute Performance:
   Color Accuracy:        0.7584  (F1: 0.4902)
   Material Accuracy:     0.8122  (F1: 0.6483)
   Condition Accuracy:    0.8077  (F1: 0.6198)
   Size Accuracy:         0.8661  (F1: 0.6942)

⭐ Overall Average Accuracy: 0.8352


/home/saksham/anaconda3/envs/model_training/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/home/saksham/anaconda3/envs/model_training/lib/python3.11/site-packages/sklearn/metrics/_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")
/home/saksham/anaconda3/envs/model_training/lib/python3.11/site-packages/sklearn/utils/multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  ys_types = set(type_of_target(x) for x in ys)
/home/saksham/anaconda3/env

In [18]:
# 📊 COMPARISON: Unified DeiT-Tiny vs Unified TinyVIT

print("\n" + "="*100)
print("🔥 MODEL COMPARISON: Unified DeiT-Tiny vs Unified TinyVIT")
print("="*100)

# ==================== Overall Metrics ====================
print(f"\n{'Metric':<35s} {'DeiT-Tiny':>18s} {'TinyVIT':>18s} {'Diff':>15s}")
print("-"*100)

# Classification Accuracy
print(f"{'Coarse Class Accuracy':<35s} {test_metrics_unified['coarse_class_acc']:>18.4f} {test_metrics_tinyvit['coarse_class_acc']:>18.4f} {test_metrics_tinyvit['coarse_class_acc']-test_metrics_unified['coarse_class_acc']:>+15.4f}")
print(f"{'Coarse Class F1':<35s} {test_metrics_unified['coarse_class_f1']:>18.4f} {test_metrics_tinyvit['coarse_class_f1']:>18.4f} {test_metrics_tinyvit['coarse_class_f1']-test_metrics_unified['coarse_class_f1']:>+15.4f}")
print()
print(f"{'Fine Class Accuracy':<35s} {test_metrics_unified['fine_class_acc']:>18.4f} {test_metrics_tinyvit['fine_class_acc']:>18.4f} {test_metrics_tinyvit['fine_class_acc']-test_metrics_unified['fine_class_acc']:>+15.4f}")
print(f"{'Fine Class F1':<35s} {test_metrics_unified['fine_class_f1']:>18.4f} {test_metrics_tinyvit['fine_class_f1']:>18.4f} {test_metrics_tinyvit['fine_class_f1']-test_metrics_unified['fine_class_f1']:>+15.4f}")

if 'fine_class_acc_hier' in test_metrics_unified and 'fine_class_acc_hier' in test_metrics_tinyvit:
    print(f"{'Fine Class (Hierarchical) Acc':<35s} {test_metrics_unified['fine_class_acc_hier']:>18.4f} {test_metrics_tinyvit['fine_class_acc_hier']:>18.4f} {test_metrics_tinyvit['fine_class_acc_hier']-test_metrics_unified['fine_class_acc_hier']:>+15.4f}")

print()
# Attribute Accuracy & F1
print(f"{'Color Accuracy':<35s} {test_metrics_unified['color_acc']:>18.4f} {test_metrics_tinyvit['color_acc']:>18.4f} {test_metrics_tinyvit['color_acc']-test_metrics_unified['color_acc']:>+15.4f}")
print(f"{'Color F1':<35s} {test_metrics_unified['color_f1']:>18.4f} {test_metrics_tinyvit['color_f1']:>18.4f} {test_metrics_tinyvit['color_f1']-test_metrics_unified['color_f1']:>+15.4f}")
print()
print(f"{'Material Accuracy':<35s} {test_metrics_unified['material_acc']:>18.4f} {test_metrics_tinyvit['material_acc']:>18.4f} {test_metrics_tinyvit['material_acc']-test_metrics_unified['material_acc']:>+15.4f}")
print(f"{'Material F1':<35s} {test_metrics_unified['material_f1']:>18.4f} {test_metrics_tinyvit['material_f1']:>18.4f} {test_metrics_tinyvit['material_f1']-test_metrics_unified['material_f1']:>+15.4f}")
print()
print(f"{'Condition Accuracy':<35s} {test_metrics_unified['condition_acc']:>18.4f} {test_metrics_tinyvit['condition_acc']:>18.4f} {test_metrics_tinyvit['condition_acc']-test_metrics_unified['condition_acc']:>+15.4f}")
print(f"{'Condition F1':<35s} {test_metrics_unified['condition_f1']:>18.4f} {test_metrics_tinyvit['condition_f1']:>18.4f} {test_metrics_tinyvit['condition_f1']-test_metrics_unified['condition_f1']:>+15.4f}")
print()
print(f"{'Size Accuracy':<35s} {test_metrics_unified['size_acc']:>18.4f} {test_metrics_tinyvit['size_acc']:>18.4f} {test_metrics_tinyvit['size_acc']-test_metrics_unified['size_acc']:>+15.4f}")
print(f"{'Size F1':<35s} {test_metrics_unified['size_f1']:>18.4f} {test_metrics_tinyvit['size_f1']:>18.4f} {test_metrics_tinyvit['size_f1']-test_metrics_unified['size_f1']:>+15.4f}")

print()
print(f"{'Overall Average Accuracy':<35s} {test_metrics_unified['avg_acc']:>18.4f} {test_metrics_tinyvit['avg_acc']:>18.4f} {test_metrics_tinyvit['avg_acc']-test_metrics_unified['avg_acc']:>+15.4f}")

print("="*100)

# Determine winner
if test_metrics_tinyvit['avg_acc'] > test_metrics_unified['avg_acc']:
    print(f"\n🏆 Winner: TinyVIT (better by {test_metrics_tinyvit['avg_acc']-test_metrics_unified['avg_acc']:.4f})")
elif test_metrics_unified['avg_acc'] > test_metrics_tinyvit['avg_acc']:
    print(f"\n🏆 Winner: DeiT-Tiny (better by {test_metrics_unified['avg_acc']-test_metrics_tinyvit['avg_acc']:.4f})")
else:
    print("\n🤝 Tie: Both models perform equally!")

print("="*100)


# ==================== Class-wise Breakdown ====================
print("\n" + "="*100)
print("📋 CLASS-WISE PERFORMANCE BREAKDOWN")
print("="*100)

from sklearn.metrics import accuracy_score, f1_score
import torch.nn.functional as F

def compute_classwise_metrics(model, loader, facet_name):
    """Compute per-class accuracy and F1 for a specific facet."""
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for images, facet_ids, captions, _ in loader:
            images = images.to(device, non_blocking=True)
            facet_batch = {f: torch.as_tensor([d[f] for d in facet_ids], device=device) for f in FACETS_UNIFIED}
            
            with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
                _, logits_attrs, _ = model(images)
            
            preds = logits_attrs[facet_name].argmax(dim=1).cpu().numpy()
            targets = facet_batch[facet_name].cpu().numpy()
            
            all_preds.extend(preds)
            all_targets.extend(targets)
    
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    
    # Get unique classes present in test set
    unique_classes = np.unique(all_targets)
    
    # Compute per-class metrics
    per_class_acc = {}
    per_class_f1 = {}
    
    for class_id in unique_classes:
        mask = all_targets == class_id
        if mask.sum() > 0:
            class_acc = (all_preds[mask] == class_id).mean()
            per_class_acc[class_id] = class_acc
            
            # F1 for this class (binary: this class vs all others)
            binary_targets = (all_targets == class_id).astype(int)
            binary_preds = (all_preds == class_id).astype(int)
            class_f1 = f1_score(binary_targets, binary_preds, zero_division=0)
            per_class_f1[class_id] = class_f1
    
    return per_class_acc, per_class_f1

# Compute class-wise metrics for coarse classes
print("\n🎯 COARSE CLASS BREAKDOWN:")
print("-"*100)

coarse_acc_deit, coarse_f1_deit = compute_classwise_metrics(model_unified, test_loader_unified, 'coarse_class')
coarse_acc_tiny, coarse_f1_tiny = compute_classwise_metrics(model_unified_tinyvit, test_loader_unified, 'coarse_class')

print(f"{'Class':<30s} {'DeiT Acc':>15s} {'DeiT F1':>15s} {'TinyVIT Acc':>15s} {'TinyVIT F1':>15s}")
print("-"*100)

for class_id in sorted(coarse_acc_deit.keys()):
    class_name = id_to_facet_value_unified['coarse_class'].get(class_id, f"Unknown_{class_id}")
    deit_acc = coarse_acc_deit.get(class_id, 0.0)
    deit_f1 = coarse_f1_deit.get(class_id, 0.0)
    tiny_acc = coarse_acc_tiny.get(class_id, 0.0)
    tiny_f1 = coarse_f1_tiny.get(class_id, 0.0)
    
    print(f"{class_name:<30s} {deit_acc:>15.4f} {deit_f1:>15.4f} {tiny_acc:>15.4f} {tiny_f1:>15.4f}")

# Compute class-wise metrics for fine classes (show top/bottom 10)
print("\n" + "="*100)
print("🔍 FINE CLASS BREAKDOWN (Top 10 Best & Worst for DeiT-Tiny):")
print("-"*100)

fine_acc_deit, fine_f1_deit = compute_classwise_metrics(model_unified, test_loader_unified, 'fine_class')
fine_acc_tiny, fine_f1_tiny = compute_classwise_metrics(model_unified_tinyvit, test_loader_unified, 'fine_class')

# Sort by DeiT accuracy
sorted_fine = sorted(fine_acc_deit.items(), key=lambda x: x[1], reverse=True)

print(f"{'Class':<40s} {'DeiT Acc':>12s} {'DeiT F1':>12s} {'TinyVIT Acc':>12s} {'TinyVIT F1':>12s}")
print("-"*100)

print("\n📈 TOP 10 BEST PERFORMING:")
for class_id, _ in sorted_fine[:10]:
    class_name = id_to_facet_value_unified['fine_class'].get(class_id, f"Unknown_{class_id}")
    deit_acc = fine_acc_deit.get(class_id, 0.0)
    deit_f1 = fine_f1_deit.get(class_id, 0.0)
    tiny_acc = fine_acc_tiny.get(class_id, 0.0)
    tiny_f1 = fine_f1_tiny.get(class_id, 0.0)
    
    print(f"{class_name:<40s} {deit_acc:>12.4f} {deit_f1:>12.4f} {tiny_acc:>12.4f} {tiny_f1:>12.4f}")

print("\n📉 BOTTOM 10 WORST PERFORMING:")
for class_id, _ in sorted_fine[-10:]:
    class_name = id_to_facet_value_unified['fine_class'].get(class_id, f"Unknown_{class_id}")
    deit_acc = fine_acc_deit.get(class_id, 0.0)
    deit_f1 = fine_f1_deit.get(class_id, 0.0)
    tiny_acc = fine_acc_tiny.get(class_id, 0.0)
    tiny_f1 = fine_f1_tiny.get(class_id, 0.0)
    
    print(f"{class_name:<40s} {deit_acc:>12.4f} {deit_f1:>12.4f} {tiny_acc:>12.4f} {tiny_f1:>12.4f}")

print("="*100)



🔥 MODEL COMPARISON: Unified DeiT-Tiny vs Unified TinyVIT

Metric                                       DeiT-Tiny            TinyVIT            Diff
----------------------------------------------------------------------------------------------------
Coarse Class Accuracy                           0.8580             0.8836         +0.0256
Coarse Class F1                                 0.7951             0.8323         +0.0372

Fine Class Accuracy                             0.8452             0.8832         +0.0380
Fine Class F1                                   0.7229             0.7912         +0.0684
Fine Class (Hierarchical) Acc                   0.8385             0.8618         +0.0233

Color Accuracy                                  0.7721             0.7584         -0.0137
Color F1                                        0.5361             0.4902         -0.0459

Material Accuracy                               0.8203             0.8122         -0.0080
Material F1                

/tmp/ipykernel_25863/2693040673.py:70: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
/tmp/ipykernel_25863/2693040673.py:70: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):


Class                                 DeiT Acc         DeiT F1     TinyVIT Acc      TinyVIT F1
----------------------------------------------------------------------------------------------------
backpack                                0.9630          0.9043          0.9074          0.9074
bathroom                                0.6486          0.6316          0.7027          0.6933
bottle                                  0.9259          0.9091          0.9630          0.9286
brush                                   0.7778          0.8750          0.8889          0.9412
calculator                              0.6667          0.8000          0.7778          0.7778
cap                                     1.0000          0.8780          0.9444          0.9444
clothing                                0.8491          0.8182          0.9057          0.8767
comb                                    0.8235          0.8750          0.9412          0.9412
cosmetics                               0.55

/tmp/ipykernel_25863/2693040673.py:70: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
/tmp/ipykernel_25863/2693040673.py:70: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):


Class                                        DeiT Acc      DeiT F1  TinyVIT Acc   TinyVIT F1
----------------------------------------------------------------------------------------------------

📈 TOP 10 BEST PERFORMING:
bathroom_bucket                                1.0000       0.8000       1.0000       0.8889
bathroom_comb                                  1.0000       1.0000       1.0000       1.0000
bathroom_razor                                 1.0000       0.8000       0.5000       0.6667
bathroom_tap                                   1.0000       0.6667       1.0000       1.0000
bathroom_toothpaste                            1.0000       0.5000       1.0000       1.0000
cap                                            1.0000       0.8780       0.9444       0.9189
clothing_socks                                 1.0000       1.0000       1.0000       0.5000
clothing_spectacles                            1.0000       1.0000       1.0000       1.0000
clothing_trousers                  

In [19]:
# Unified classifier that accepts path, PIL.Image, or numpy array
import tempfile
import numpy as np
from PIL import Image

def _ensure_image_path(image_input) -> str:
    # Returns a temp file path if input is PIL/numpy; or the original string if it's already a path
    if isinstance(image_input, str):
        return image_input
    if isinstance(image_input, Image.Image):
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.jpg')
        image_input.save(tmp.name)
        return tmp.name
    if isinstance(image_input, np.ndarray):
        img = Image.fromarray(image_input.astype(np.uint8))
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.jpg')
        img.save(tmp.name)
        return tmp.name
    raise ValueError("Unsupported image input type; provide str path, PIL.Image, or numpy array.")

@torch.no_grad()
def classify_image_unified(image_path, model: nn.Module, show_image: bool = True, use_hierarchy: bool = True) -> Dict:
    """
    Classify image using unified model (classes as attributes).
    Accepts a file path, PIL.Image, or numpy array.
    """
    actual_path = _ensure_image_path(image_path)
    if not os.path.exists(actual_path):
        raise FileNotFoundError(f"Image not found: {actual_path}")

    # Load and transform image
    img = Image.open(actual_path).convert("RGB")
    img_tensor = val_transforms(img).unsqueeze(0).to(device)

    # Get predictions
    model.eval()
    with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
        feats, logits_attrs, img_emb = model(img_tensor)

    # First, predict coarse class
    coarse_logits = logits_attrs["coarse_class"]
    coarse_pred_id = coarse_logits.argmax(dim=1).item()
    coarse_pred_value = id_to_facet_value_unified["coarse_class"][coarse_pred_id]
    coarse_confidence = torch.softmax(coarse_logits, dim=1)[0, coarse_pred_id].item()

    # Predict fine class with hierarchical constraint
    fine_logits = logits_attrs["fine_class"]
    fine_probs = torch.softmax(fine_logits, dim=1)[0]

    if use_hierarchy and coarse_pred_value in coarse_to_fine_mapping:
        valid_fine_classes = coarse_to_fine_mapping[coarse_pred_value]
        valid_fine_ids = [facet_to_id_unified["fine_class"][fc] for fc in valid_fine_classes]
        masked_probs = torch.zeros_like(fine_probs)
        masked_probs[valid_fine_ids] = fine_probs[valid_fine_ids]
        if masked_probs.sum() > 0:
            masked_probs = masked_probs / masked_probs.sum()
        fine_pred_id = masked_probs.argmax().item()
        fine_confidence = masked_probs[fine_pred_id].item()
    else:
        fine_pred_id = fine_probs.argmax().item()
        fine_confidence = fine_probs[fine_pred_id].item()

    fine_pred_value = id_to_facet_value_unified["fine_class"][fine_pred_id]

    # Decode other attributes
    all_predictions = {
        "coarse_class": {"value": coarse_pred_value, "confidence": coarse_confidence},
        "fine_class": {"value": fine_pred_value, "confidence": fine_confidence}
    }
    for facet in ["color", "material", "condition", "size"]:
        pred_id = logits_attrs[facet].argmax(dim=1).item()
        pred_value = id_to_facet_value_unified[facet][pred_id]
        confidence = torch.softmax(logits_attrs[facet], dim=1)[0, pred_id].item()
        all_predictions[facet] = {"value": pred_value, "confidence": confidence}

    if show_image:
        try:
            import matplotlib.pyplot as plt
            plt.figure(figsize=(8, 6))
            plt.imshow(img)
            plt.axis('off')
            title = f"Coarse: {all_predictions['coarse_class']['value']} | Fine: {all_predictions['fine_class']['value']}"
            plt.title(title)
            plt.tight_layout()
            plt.show()
        except ImportError:
            pass

    return {
        "image_path": actual_path,
        "coarse_class": all_predictions["coarse_class"],
        "fine_class": all_predictions["fine_class"],
        "attributes": {
            "color": all_predictions["color"],
            "material": all_predictions["material"],
            "condition": all_predictions["condition"],
            "size": all_predictions["size"]
        },
        "embedding": img_emb.cpu().numpy()[0]
    }


In [20]:
# Guarded optimizer/scaler for legacy multi-head path (safe if undefined)
if 'model' in globals():
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
    frozen = False
else:
    optimizer = None
    scaler = None
    frozen = False

def set_backbone_trainable(module: nn.Module, trainable: bool):
    for p in module.parameters():
        p.requires_grad = trainable


In [21]:
# Prediction function for unified model with hierarchical constraint
@torch.no_grad()
def classify_image_unified(image_path: str, model: nn.Module, show_image: bool = True, use_hierarchy: bool = True) -> Dict:
    """
    Classify image using unified model (classes as attributes).
    
    Args:
        image_path: Path to image file
        model: Trained unified model
        show_image: Whether to display the image
        use_hierarchy: Whether to apply hierarchical constraint (coarse -> fine)
    
    Returns:
        Dictionary with predictions
    """
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")
    
    # Load and transform image
    img = Image.open(image_path).convert("RGB")
    img_tensor = val_transforms(img).unsqueeze(0).to(device)
    
    # Get predictions
    model.eval()
    with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
        feats, logits_attrs, img_emb = model(img_tensor)
    
    # First, predict coarse class
    coarse_logits = logits_attrs["coarse_class"]
    coarse_pred_id = coarse_logits.argmax(dim=1).item()
    coarse_pred_value = id_to_facet_value_unified["coarse_class"][coarse_pred_id]
    coarse_confidence = torch.softmax(coarse_logits, dim=1)[0, coarse_pred_id].item()
    
    # Predict fine class with hierarchical constraint
    fine_logits = logits_attrs["fine_class"]
    fine_probs = torch.softmax(fine_logits, dim=1)[0]
    
    if use_hierarchy and coarse_pred_value in coarse_to_fine_mapping:
        # Get valid fine class IDs for this coarse class
        valid_fine_classes = coarse_to_fine_mapping[coarse_pred_value]
        valid_fine_ids = [facet_to_id_unified["fine_class"][fc] for fc in valid_fine_classes]
        
        # Mask invalid fine classes
        masked_probs = torch.zeros_like(fine_probs)
        masked_probs[valid_fine_ids] = fine_probs[valid_fine_ids]
        
        # Renormalize probabilities over valid classes
        if masked_probs.sum() > 0:
            masked_probs = masked_probs / masked_probs.sum()
        
        fine_pred_id = masked_probs.argmax().item()
        fine_confidence = masked_probs[fine_pred_id].item()
    else:
        # No hierarchy constraint
        fine_pred_id = fine_probs.argmax().item()
        fine_confidence = fine_probs[fine_pred_id].item()
    
    fine_pred_value = id_to_facet_value_unified["fine_class"][fine_pred_id]
    
    # Decode other attributes
    all_predictions = {
        "coarse_class": {
            "value": coarse_pred_value,
            "confidence": coarse_confidence
        },
        "fine_class": {
            "value": fine_pred_value,
            "confidence": fine_confidence
        }
    }
    
    for facet in ["color", "material", "condition", "size"]:
        pred_id = logits_attrs[facet].argmax(dim=1).item()
        pred_value = id_to_facet_value_unified[facet][pred_id]
        confidence = torch.softmax(logits_attrs[facet], dim=1)[0, pred_id].item()
        all_predictions[facet] = {
            "value": pred_value,
            "confidence": confidence
        }
    
    # Display image if requested
    if show_image:
        try:
            import matplotlib.pyplot as plt
            plt.figure(figsize=(8, 6))
            plt.imshow(img)
            plt.axis('off')
            title = f"Coarse: {all_predictions['coarse_class']['value']} | Fine: {all_predictions['fine_class']['value']}"
            plt.title(title)
            plt.tight_layout()
            plt.show()
        except ImportError:
            print("matplotlib not available; skipping image display")
    
    # Prepare results
    results = {
        "image_path": image_path,
        "coarse_class": all_predictions["coarse_class"],
        "fine_class": all_predictions["fine_class"],
        "attributes": {
            "color": all_predictions["color"],
            "material": all_predictions["material"],
            "condition": all_predictions["condition"],
            "size": all_predictions["size"]
        },
        "embedding": img_emb.cpu().numpy()[0]
    }
    
    return results


# Pretty print function
def print_predictions_unified(results: Dict):
    """Pretty print unified model predictions."""
    print(f"\n{'='*60}")
    print(f"Image: {os.path.basename(results['image_path'])}")
    print(f"{'='*60}")
    print(f"\n📦 COARSE CLASS: {results['coarse_class']['value'].upper()}")
    print(f"   Confidence: {results['coarse_class']['confidence']:.2%}")
    print(f"\n🔍 FINE-GRAINED CLASS: {results['fine_class']['value'].upper()}")
    print(f"   Confidence: {results['fine_class']['confidence']:.2%}")
    print(f"\n🏷️  ATTRIBUTES:")
    for facet, pred in results["attributes"].items():
        print(f"   {facet.capitalize():12s}: {pred['value']:15s} (confidence: {pred['confidence']:.2%})")
    print(f"{'='*60}\n")


print("✅ Prediction functions ready for unified model")


✅ Prediction functions ready for unified model


In [ ]:
# 🖼️ Gradio Interface for Image Classification

import gradio as gr

@torch.no_grad()
def predict_image_gradio(image: Image.Image, model_name: str):
    """Run inference on an image for Gradio interface."""
    if image is None:
        return {"error": "Please upload an image"}
    
    # Select model
    model = model_unified if model_name == "DeiT-Tiny" else model_unified_tinyvit
    
    # Transform image
    img_tensor = val_transforms(image).unsqueeze(0).to(device)
    
    # Run inference
    with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
        _, logits_attrs, _ = model(img_tensor)
    
    # Get predictions
    predictions = {}
    for facet in FACETS_UNIFIED:
        logits = logits_attrs[facet]
        probs = F.softmax(logits, dim=1)
        pred_id = probs.argmax(dim=1).item()
        confidence = probs[0, pred_id].item()
        pred_value = id_to_facet_value_unified[facet][pred_id]
        
        # Get top 3 predictions
        top3_probs, top3_ids = torch.topk(probs[0], min(3, len(probs[0])))
        top3_values = [id_to_facet_value_unified[facet][idx.item()] for idx in top3_ids]
        top3_confidences = [prob.item() for prob in top3_probs]
        
        predictions[facet] = {
            "predicted": pred_value,
            "confidence": f"{confidence:.2%}",
            "top3": list(zip(top3_values, [f"{c:.2%}" for c in top3_confidences]))
        }
    
    return predictions


def format_predictions_gradio(predictions):
    """Format predictions for display."""
    if "error" in predictions:
        return predictions["error"]
    
    output = []
    output.append("## Classification Results\n")
    
    # Class predictions
    output.append("### 🏷️ Class Predictions")
    output.append(f"**Coarse Class:** {predictions['coarse_class']['predicted']} ({predictions['coarse_class']['confidence']})")
    output.append(f"**Fine Class:** {predictions['fine_class']['predicted']} ({predictions['fine_class']['confidence']})")
    
    # Attributes
    output.append("\n### 🎨 Attributes")
    for attr in ["color", "material", "condition", "size"]:
        pred = predictions[attr]
        output.append(f"**{attr.capitalize()}:** {pred['predicted']} ({pred['confidence']})")
        if len(pred['top3']) > 1:
            alternatives = ", ".join([f"{val} ({conf})" for val, conf in pred['top3'][1:]])
            output.append(f"  *Alternatives: {alternatives}*")
    
    return "\n".join(output)


def classify_image_interface(image, model_choice: str) -> str:
    """Main classification function for Gradio."""
    try:
        predictions = predict_image_gradio(image, model_choice)
        return format_predictions_gradio(predictions)
    except Exception as e:
        return f"Error: {str(e)}"


# Create Gradio interface
with gr.Blocks(title="Image Classification - DeiT & TinyVIT") as demo:
    gr.Markdown("# 🖼️ Image Classification Interface")
    gr.Markdown(f"Upload an image to classify it using DeiT-Tiny or TinyVIT models trained on {DATASET_TAG}.")
    
    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="Upload Image")
            model_choice = gr.Radio(
                choices=["DeiT-Tiny", "TinyVIT"],
                value="DeiT-Tiny",
                label="Select Model"
            )
            classify_btn = gr.Button("Classify Image", variant="primary")
        
        with gr.Column():
            output = gr.Markdown(label="Predictions")
    
    classify_btn.click(
        fn=classify_image_interface,
        inputs=[image_input, model_choice],
        outputs=output
    )
    
    gr.Markdown("### 📊 Model Information")
    gr.Markdown(f"- **Dataset:** {DATASET_TAG}")
    gr.Markdown("- **Models:** DeiT-Tiny (deit_tiny_patch16_224) and TinyVIT (tiny_vit_5m_224.dist_in22k_ft_in1k)")
    gr.Markdown(f"- **Coarse Classes:** {len(id_to_facet_value_unified['coarse_class'])}")
    gr.Markdown(f"- **Fine Classes:** {len(id_to_facet_value_unified['fine_class'])}")
    gr.Markdown("- **Attributes:** Color, Material, Condition, Size")

# Launch the interface
print("🚀 Launching Gradio interface...")
demo.launch(share=False, server_name="0.0.0.0", server_port=7860)


🚀 Launching Gradio interface...
* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.


/tmp/ipykernel_25863/8145029.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
/tmp/ipykernel_25863/8145029.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
/tmp/ipykernel_25863/8145029.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
/tmp/ipykernel_25863/8145029.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
